In [3]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu" # 장치 선택 Convention

In [4]:
# GPU를 쓰려면 텐서와 모델을 같은 장치로 옮기면 됩니다.

# 1) 생성 시 장치 지정
x = torch.randn(3, 3, device=device)

# 2) 기존 텐서를 이동 (.to 또는 .cuda)
y = torch.randn(3, 3)
y = y.to(device) # to 함수 호출을 통해 장치에 데이터를 저장하고 해당 레퍼런스를 받음.

# 연산하는 두 텐서는 같은 장치에 있어야 함.
z = x + y
print(z.device)

# GPU -> CPU (NumPy 변환이나 출력 전에)
z_cpu = z.cpu()

cuda:0


# Autograd: 자동 미분
**Autograd** 는 PyTorch가 자동으로 기울기를 계산해주는 엔진이다. 신경망 학습의 핵심인 **역전파 알고리즘** 이 위에서 동작한다.

## 손실계산과, .backward()의 주의사항
손실 값은 스칼라 값이다. 그래서 .backward()는 기본적으로 스칼라 값에서 호출 가능하며, 다차원 텐서에서의 호출은 gradient 인자를 통해 grad_fn을 넘겨야 한다.

각 텐서는 자신이 어떤 연산으로 만들어졌는지(.grad_fn)을 기억하며, 이 기록을 따라 연쇄법칙으로 기울기를 전파한다.

In [12]:
# requires_grad & backward
# requires_grad=True로 설정하면 PyTorch가 해당 텐서에 대한 연산을 추적해 계산 그래프를 만든다.
# .backward()를 호출하면 그래프를 거꾸로 따라가며 기울기를 계산한다.

# x에 대한 기울기를 추적
x = torch.tensor(3.0, requires_grad=True)

y = x**2 + 2*x + 1

# dy/dx를 계산
y.backward()

print(x.grad) # dy/dx = 2x+2 = 8

# 벡터 입력에 대해서도 똑같이 동작
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x**2).sum()
y.backward()
print(x.grad) # dy/dx_i = 2*x_i -> tensor([2, 4, 6])

# 계산 그래프와 grad_fn
x = torch.tensor(2.0, requires_grad = True)
y = x * 3
z = y + 1
print(y.grad_fn)
print(z.grad_fn)

tensor(8.)
tensor([2., 4., 6.])


### 기울기 추적 끄기(중요)
추론(inference) 시에는 기울기가 필요 없으므로, 추적을 꺼서 메모리와 속도를 아낀다.

In [13]:
# torch.no_grad() 컨텍스트 안에서 코드 실행
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x * 2
    print(y.requires_grad) # False


# detach로 그래프에서 분리
z = (x * 2).detach()
print(z.requires_grad)

False
False


### 기울기 누적과 초기화(중요)
PyTorch는 .grad에 기울기를 누적(+=) 한다. 따라서 매 반복마다(Epoch) 초기화 해야 한다. 
<mark>--> 누적을 하면 역전파를 통한 새 기울기를 덮어쓰는게 아니라 기존 값에 더하게 된다.</mark> 그래서 초기화를 하고 새 기울기를 더해야 한다.
지우지 않으면 지난 배치의 기울기가 계속 섞여 들어간다

In [ ]:
x = torch.tensor(1.0, requires_grad=True)

for step in range(3):
    y = x**2 # dy/dx = 2x -> 2
    y.backward()
    print(f"step {step}: grad = {x.grad}") # 2, 4, 6 누적 됨.
    x.grad.zero_() # 초기화 하지 않으면 계속 더해짐

step 0: grad = 2.0
step 1: grad = 2.0
step 2: grad = 2.0


# Autograd를 통해 직접 구현하는 경사하강법
예시 $f(x) = (x-3)^2$의 최솟값($x=3$)을 경사하강법으로 찾는 예제

In [ ]:
x = torch.tensor(0.0, requires_grad = True)
lr = 0.1 # 학습률(eta)

for step in range(30):
    loss = (x - 3) ** 2     # 최소화 할 함수 (손실 함수)
    loss.backward()         # 기울기 계산
    
    with torch.no_grad():       # 파라미터 갱신은 추적하지 않음.
        x -= lr * x.grad        # 경사 하강법 x <- x - (eta)*(dL/dx) L은 손실함수
        x.grad.zero_()          # 기울기 초기화
        
    if step % 10 == 0:
        print(f"step {step:2d}: x={x.item():.4f}, loss={loss.item():.4f}")
        
print(f"최종 x = {x.item():.4f} (정답: 3.0)")

step  0: x=0.6000, loss=9.0000
step 10: x=2.7423, loss=0.1038
step 20: x=2.9723, loss=0.0012
최종 x = 2.9963 (정답: 3.0)
